In [13]:
!pip install -q openai

In [14]:
import json, os, re, time, random
from google.colab import userdata
from openai import OpenAI

API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4o"

In [15]:
# 1. 시드데이터 로드
with open('PEP_SEED_DATA.json', encoding='utf-8') as f:
    seed_data = json.load(f)

print(f"시드 개수: {len(seed_data)}")
existing_ids = set(item['id'] for item in seed_data)

시드 개수: 30


In [16]:
PEP_DESCRIPTION_MAP = {
    "PEP-03-01": {
        "id": ["PEP-03-01"],
        "section_title": "사업목적",
        "description": "제안요청서의 추진배경 및 필요성을 근거로 사업 수행의 배경을 기술하는 항목이다. RFP의 추진배경·필요성이 PEP 추진배경에 누락 없이 반영되었는지를 평가한다.",
        "standard_structure": ["추진배경"],
        "quality": ["완전성"],
        "rfp_mapping": ["RFP-01-02"],
        "rfp_mapping_note": "추진배경·필요성"
    },
    "PEP-03-02": {
        "id": ["PEP-03-02"],
        "section_title": "사업목적",
        "description": "제안요청서의 추진목표를 근거로 사업 목적을 기술하는 항목이다. RFP의 추진목표가 PEP 목적에 누락 없이 반영되었는지를 평가한다.",
        "standard_structure": ["목적"],
        "quality": ["완전성"],
        "rfp_mapping": ["RFP-03-01"],
        "rfp_mapping_note": "추진목표"
    },
    "PEP-04-01": {
        "id": ["PEP-04-01"],
        "section_title": "사업범위",
        "description": "개발대상업무를 기술하는 항목이다. RFP 기능요구사항이 PEP 개발대상업무에 누락 없이 반영되었는지를 평가한다.",
        "standard_structure": ["개발대상업무"],
        "quality": ["완전성"],
        "rfp_mapping": ["RFP-04-04-01"],
        "rfp_mapping_note": "기능요구사항"
    },
    "PEP-04-02": {
        "id": ["PEP-04-02"],
        "section_title": "사업범위",
        "description": "개발 및 운영환경을 기술하는 항목이다. 소프트웨어, 하드웨어, 네트워크 등 기술 환경이 RFP 시스템 장비구성 요구사항 및 현행 시스템 현황과 모순 없는지 평가한다.",
        "standard_structure": ["개발 및 운영환경", "소프트웨어", "하드웨어", "네트워크"],
        "quality": ["정확성"],
        "rfp_mapping": ["RFP-04-04-02", "RFP-02-02"],
        "rfp_mapping_note": "시스템 장비구성 요구사항, 시스템현황"
    },
    "PEP-04-03": {
        "id": ["PEP-04-03"],
        "section_title": "사업범위",
        "description": "인터페이스, 표준화, 업무절차재구축, 초기자료구축 등 기타 사업범위를 기술하는 항목이다. RFP의 인터페이스 및 제약사항 요구가 누락 없이 반영되었는지를 평가한다.",
        "standard_structure": ["기타", "인터페이스 관련사항", "표준화", "업무절차재구축", "초기자료구축"],
        "quality": ["완전성"],
        "rfp_mapping": ["RFP-04-04-04", "RFP-04-04-09"],
        "rfp_mapping_note": "인터페이스, 제약사항 요구사항"
    },
    "PEP-05-01": {
        "id": ["PEP-05-01"],
        "section_title": "사업추진체계",
        "description": "총괄추진체계를 기술하는 항목이다. RFP의 추진체계 요구가 발주기관, 이용기관, 사업자 등 역할 구조에 누락 없이 반영되었는지를 평가한다.",
        "standard_structure": ["총괄추진체계"],
        "quality": ["완전성"],
        "rfp_mapping": ["RFP-03-02"],
        "rfp_mapping_note": "추진체계"
    },
    "PEP-05-02": {
        "id": ["PEP-05-02"],
        "section_title": "사업추진체계",
        "description": "사업자 추진체계와 역할 및 업무분장을 기술하는 항목이다. RFP 추진체계 및 프로젝트 관리 요구사항이 사업자 조직도와 업무분장에 누락 없이 반영되었는지를 평가한다.",
        "standard_structure": ["사업자 추진체계", "역할 및 업무분장"],
        "quality": ["완전성"],
        "rfp_mapping": ["RFP-03-02", "RFP-04-04-10"],
        "rfp_mapping_note": "추진체계, 프로젝트 관리 요구사항"
    },
    "PEP-06": {
        "id": ["PEP-06"],
        "section_title": "사업추진절차",
        "description": "사업 단계, 세그먼트, 단위업무, 수행업무, 산출물을 기술하는 항목이다. RFP 상세요구사항 전 영역이 단계별 수행업무로 누락 없이 반영되었는지를 평가한다.",
        "standard_structure": ["사업 단계", "세그먼트", "단위업무", "수행업무", "산출물"],
        "quality": ["완전성"],
        "rfp_mapping": [
            "RFP-04-04-01", "RFP-04-04-02", "RFP-04-04-03",
            "RFP-04-04-04", "RFP-04-04-05", "RFP-04-04-06",
            "RFP-04-04-07", "RFP-04-04-08", "RFP-04-04-09",
            "RFP-04-04-10", "RFP-04-04-11"
        ],
        "rfp_mapping_note": "제안요청내용 상세요구사항"
    },
    "PEP-07": {
        "id": ["PEP-07"],
        "section_title": "산출물계획",
        "description": "산출물명, 제출일정, 제출부수 등 산출물 제출계획을 기술하는 항목이다. RFP 요구 산출물이 누락 없이 포함되었는지와 사업추진절차의 단위업무·일정·산출물 간 연결이 추적 가능한지 평가한다.",
        "standard_structure": ["산출물명", "제출일정", "제출부수"],
        "quality": ["완전성", "추적성"],
        "rfp_mapping": [
            "RFP-04-04-01", "RFP-04-04-02", "RFP-04-04-03",
            "RFP-04-04-04", "RFP-04-04-05", "RFP-04-04-06",
            "RFP-04-04-07", "RFP-04-04-08", "RFP-04-04-09",
            "RFP-04-04-10", "RFP-04-04-11"
        ],
        "rfp_mapping_note": "제안요청내용 상세요구, 사업추진절차"
    },
    "PEP-10": {
        "id": ["PEP-10"],
        "section_title": "보고계획",
        "description": "주간보고, 월간보고, 단계별 보고, 품질보증활동보고, 위험관리현황보고 등 전체 보고계획을 기술하는 항목이다. RFP 프로젝트 관리 요구사항의 보고 유형이 누락 없이 반영되었는지를 평가한다.",
        "standard_structure": ["주간보고", "월간보고", "단계별보고", "품질보증활동보고", "위험관리현황보고"],
        "quality": ["완전성"],
        "rfp_mapping": ["RFP-04-04-10"],
        "rfp_mapping_note": "프로젝트 관리 요구사항"
    },
    "PEP-11-01": {
        "id": ["PEP-11-01"],
        "section_title": "표준화계획",
        "description": "표준화 항목에 맞는 사업수행내역을 기술하는 항목이다. RFP 요구사항 총괄표의 표준화 요구가 누락 없이 반영되었는지와 확인 가능한 방식으로 제시되었는지를 평가한다.",
        "standard_structure": ["표준화 항목", "표준화 항목에 맞는 사업수행내역"],
        "quality": ["완전성", "검증가능성"],
        "rfp_mapping": ["RFP-04-03"],
        "rfp_mapping_note": "요구사항 총괄표"
    },
    "PEP-11-02": {
        "id": ["PEP-11-02"],
        "section_title": "표준화계획",
        "description": "정보화기반표준 준수 계획을 기술하는 항목이다. 적용 기준과 확인 방법이 측정 가능하거나 확인 가능한 형태로 제시되었는지를 평가한다.",
        "standard_structure": ["정보화기반표준"],
        "quality": ["검증가능성"],
        "rfp_mapping": [],
        "rfp_mapping_note": ""
    },
    "PEP-11-03": {
        "id": ["PEP-11-03"],
        "section_title": "표준화계획",
        "description": "공공기관 DB표준화 지침 준수 계획을 기술하는 항목이다. 명명규칙, 코드체계, 데이터 표준 점검 방식이 확인 가능한 형태로 제시되었는지를 평가한다.",
        "standard_structure": ["공공기관 DB표준화 지침"],
        "quality": ["검증가능성"],
        "rfp_mapping": [],
        "rfp_mapping_note": ""
    },
    "PEP-11-04": {
        "id": ["PEP-11-04"],
        "section_title": "표준화계획",
        "description": "전자정부 웹사이트 품질관리 지침 준수 계획을 기술하는 항목이다. 웹표준, 웹접근성, 호환성 등의 점검 기준이 확인 가능한 형태로 제시되었는지를 평가한다.",
        "standard_structure": ["전자정부 웹사이트 품질관리 지침"],
        "quality": ["검증가능성"],
        "rfp_mapping": [],
        "rfp_mapping_note": ""
    },
    "PEP-12": {
        "id": ["PEP-12"],
        "section_title": "품질관리계획",
        "description": "품질목표, 추진체계, 내용, 품질보증절차를 기술하는 항목이다. RFP 품질 요구사항이 누락 없이 반영되었는지, 수치·기준이 정확한지, 검증 가능한 목표와 절차로 제시되었는지를 평가한다.",
        "standard_structure": ["품질목표", "품질 추진체계", "품질보증 절차"],
        "quality": ["검증가능성", "정확성", "완전성"],
        "rfp_mapping": ["RFP-04-04-08"],
        "rfp_mapping_note": "품질 요구사항"
    },
    "PEP-13": {
        "id": ["PEP-13"],
        "section_title": "위험관리계획",
        "description": "위험관리목표, 추진체계, 절차를 기술하는 항목이다. RFP 프로젝트 관리 요구사항의 위험관리 항목이 누락 없이 반영되었는지와 관리 기준이 확인 가능한 형태로 제시되었는지를 평가한다.",
        "standard_structure": ["위험관리목표", "추진체계", "절차"],
        "quality": ["검증가능성", "완전성"],
        "rfp_mapping": ["RFP-04-04-10"],
        "rfp_mapping_note": "프로젝트 관리 요구사항"
    },
    "PEP-14": {
        "id": ["PEP-14"],
        "section_title": "보안대책",
        "description": "문서보안, 통신보안, 시스템보안, 개인정보보호 대책을 기술하는 항목이다. RFP 보안요구사항이 누락 없이 반영되었는지, 보안 규격이 정확한지, 점검 기준이 검증 가능한 형태로 제시되었는지를 평가한다.",
        "standard_structure": ["문서·통신보안", "시스템보안", "개인정보보호", "개발자교육"],
        "quality": ["검증가능성", "정확성", "완전성"],
        "rfp_mapping": ["RFP-04-04-07"],
        "rfp_mapping_note": "보안요구사항"
    },
    "PEP-15": {
        "id": ["PEP-15"],
        "section_title": "교육계획",
        "description": "사업 완료 이전과 이후 제공할 교육의 과목, 일정, 대상, 기간, 교육내용 및 지원 사항을 기술하는 항목이다. RFP 지원요구사항이 누락 없이 정확하게 반영되었는지를 평가한다.",
        "standard_structure": ["교육 과목", "교육 일정", "교육 대상", "교육 기간", "교육내용", "지원 사항"],
        "quality": ["완전성", "정확성"],
        "rfp_mapping": ["RFP-04-04-11"],
        "rfp_mapping_note": "지원요구사항"
    },
    "PEP-16": {
        "id": ["PEP-16"],
        "section_title": "발주기관 협조요청사항",
        "description": "사업자의 입장에서 사업수행을 위해 발주기관이 조치해야 하는 사항을 기술하는 항목이다. 출입, 자료 제공, API 제공, UAT 협조 등 요청사항이 확인 가능한 수준으로 제시되었는지를 평가한다.",
        "standard_structure": ["출입 협조", "자료 제공", "API 제공", "UAT 협조", "기타 발주기관 조치사항"],
        "quality": ["검증가능성"],
        "rfp_mapping": [],
        "rfp_mapping_note": ""
    }
}

print(f"PEP_DESCRIPTION_MAP 항목 수: {len(PEP_DESCRIPTION_MAP)}")

PEP_DESCRIPTION_MAP 항목 수: 19


In [17]:
DEFECT_PREFIX = {"완전성": "완전", "정확성": "정확", "검증가능성": "검증", "추적성": "추적"}

def build_combos():
    combos = []
    for code_, desc in PEP_DESCRIPTION_MAP.items():
        quality = desc.get('quality') or []
        if not quality:
            continue
        combos.append({"desc_code": code_, "target_criteria": list(quality), "target_defect_type": "충족"})
        for c in quality:
            prefix = DEFECT_PREFIX[c]
            combos.append({"desc_code": code_, "target_criteria": list(quality), "target_defect_type": f"{prefix}불가"})
            combos.append({"desc_code": code_, "target_criteria": list(quality), "target_defect_type": f"{prefix}검토"})
    return combos

combos = build_combos()
random.seed(42)
random.shuffle(combos)
print(f"조합 개수: {len(combos)}")

조합 개수: 73


In [18]:
TARGET_TOTAL = 150
BATCH_N = 5
MAX_CALLS = 60

In [19]:
SYSTEM_PROMPT = """당신은 과업수행계획서(PEP) 평가 학습데이터를 증강하는 생성기입니다.
목표는 기존 시드데이터와 동일한 스키마·품질 기준으로 새로운 {"id", "input", "output"} 예시를 생성하는 것입니다.

이 프롬프트에서는 대제목별 평가기준 매트릭스를 직접 사용하지 않습니다.
평가기준 조합과 RFP 매핑은 코드에서 관리되는 target_description 객체가 이미 결정합니다.

[입력]
- target_description: 코드에서 관리되는 PEP_DESCRIPTION_MAP 중 하나의 description 객체
  - id: 평가 대상 PEP 코드 목록
  - section_title: 과업수행계획서 대제목
  - description: 해당 항목의 평가 목적 설명
  - standard_structure: 해당 항목에서 확인해야 할 하위 구성요소 목록
  - quality: 해당 항목에 허용된 평가기준 목록
  - rfp_mapping: 대조할 RFP 코드 목록
  - rfp_mapping_note: 사람이 이해하기 위한 RFP 매핑 참고 설명

- target_criteria: 이번 예시에서 실제로 다룰 평가기준
  - 반드시 target_description.quality 안에 포함된 기준만 사용합니다.
  - 예: ["완전성"], ["완전성", "정확성"], ["검증가능성", "완전성"]

- target_defect_type: 목표 결함 유형
  - 충족
  - 완전불가
  - 정확불가
  - 검증불가
  - 추적불가
  - 완전검토
  - 정확검토
  - 검증검토
  - 추적검토

[description 사용 규칙]
- 생성 요청 시 전체 매트릭스가 아니라 target_description 객체가 입력됩니다.
- target_description은 코드에서 관리되는 PEP_DESCRIPTION_MAP에서 선택된 값입니다.
- 생성기는 target_description에 포함된 PEP 코드, 대제목, 하위 구성요소, 평가기준, RFP 매핑만 사용합니다.
- target_description.quality에 없는 평가기준은 임의로 추가하지 않습니다.
- target_description.rfp_mapping에 없는 RFP 항목은 임의로 추가하지 않습니다.
- target_description의 description, standard_structure, rfp_mapping_note는 생성 범위 설정용 메타데이터이며, 최종 코멘트의 원문 근거로 인용하지 않습니다.

[평가기준명 정규화]
평가기준명은 반드시 아래 네 가지 중 하나만 사용합니다.
- 완전성
- 정확성
- 검증가능성
- 추적성

정규화 규칙:
- "완전"은 "완전성"으로 변환합니다.
- "정확"은 "정확성"으로 변환합니다.
- "검증"은 "검증가능성"으로 변환합니다.
- "연결"은 "추적성"으로 변환합니다.
- "코드" 또는 "-"는 LLM 평가기준이 아니므로 생성 대상에서 제외합니다.

[생성 절차 — 반드시 순서대로]

1단계. 가상의 공공 정보화사업 설정
- 지자체/교육청/공공기관 + 사업 유형을 새로 조합합니다.
- 기존 시드와 대상기관 수, 사업 유형, 소재 도메인이 겹치지 않게 합니다.
- 사업 도메인 힌트가 주어진 경우 그 범위 안에서 새 사업을 만듭니다.

2단계. rfp_excerpt 작성
- rfp_excerpt는 제안요청서 원문처럼 작성합니다.
- target_description.rfp_mapping에 해당하는 RFP 요구사항만 작성합니다.
- target_description.rfp_mapping에 없는 RFP 항목을 임의로 추가하지 않습니다.
- 완전성을 다룰 경우 3~5개의 체크 가능한 요구 항목을 명시적으로 나열합니다.
- 정확성을 다룰 경우 나중에 pep_excerpt에서 왜곡할 수 있는 구체 수치·규격·버전·등급·기간을 반드시 포함합니다.
- 검증가능성을 다룰 경우 측정 기준, 정량 목표, 점검 방법, 기준일, 제출 근거 중 하나 이상을 포함합니다.
- 추적성을 다룰 경우 요구사항이 과업, Task, 일정, 산출물로 연결되어야 함을 명시합니다.
- 검토 시나리오를 만들 경우 "필요한 경우", "~할 수 있다", "별첨에 따른다", "붙임에 명시한다" 같은 조건부 또는 외부자료 의존 문구를 사용합니다.

3단계. pep_excerpt 작성 — target_defect_type에 맞게 설계
- pep_excerpt는 과업수행계획서 원문처럼 작성합니다.
- target_description.id에 해당하는 PEP 항목만 작성합니다.
- target_description.standard_structure에 포함된 하위 구성요소를 중심으로 작성합니다.
- target_criteria와 target_defect_type에 맞게 충족/불가/검토 상황을 의도적으로 설계합니다.

[충족]
- RFP 요구 항목·수치·구조가 PEP에 빠짐없이, 모순 없이, 측정 가능하게 반영되도록 작성합니다.

[완전불가]
- RFP 요구 항목 중 정확히 1개만 PEP 어디에도 등장하지 않게 만듭니다.
- 나머지 요구 항목은 정상 반영합니다.
- 단일값 필드의 누락이 아니라 요구 기능, 역할, 절차, 보고유형, 산출물 등 내용 항목의 누락으로 만듭니다.

[정확불가]
- RFP의 구체 수치·규격·버전·등급·기간을 PEP에서 다르게 기재합니다.
- 예: TLS 1.3 요구 → PEP는 TLS 1.2 적용, 2인 승인 요구 → PEP는 1인 승인.

[검증불가]
- RFP가 정량 목표나 확인 가능한 기준을 요구했는데 PEP는 "적절히", "최선을 다해", "원활히 협의하여", "필요한 범위에서", "우수한 수준으로" 같은 모호 표현으로만 작성합니다.
- 구조나 항목은 존재하되 측정 기준이 없도록 만듭니다.

[추적불가]
- 특정 Task는 기재하되 해당 Task의 산출물 또는 일정 중 하나 이상을 누락합니다.
- 다른 Task들은 산출물·일정이 정상적으로 연결되도록 하여 특정 Task의 연결 단절이 드러나게 합니다.

[완전검토]
- 빠진 항목이 RFP에서 조건부 문구로 되어 있어 필수 여부가 모호하게 만들거나, RFP 자체가 붙임/별첨을 참조하지만 해당 붙임이 입력에 없도록 만듭니다.

[정확검토]
- PEP에 수치나 규격은 있으나 RFP, 계약서, 현황자료, 별첨이 미첨부되어 일치 여부를 확인할 수 없게 만듭니다.
- 또는 RFP가 "최신 버전", "관련 지침"처럼 비교 기준값을 구체적으로 제시하지 않아 정확성 판단 대상이 아님에 가까운 케이스를 만듭니다.

[검증검토]
- 같은 평가 범위 안에서 일부 하위항목은 정량 기준을 제시하고, 일부 하위항목은 모호 표현만 사용합니다.

[추적검토]
- 여러 Task 중 일부는 산출물·일정이 명확히 매핑되고, 일부는 제출일정 또는 산출물 확인이 입력에 없어 전체 추적 가능 여부가 불명확하게 만듭니다.

[COMP/VER 경계]
- 요구된 구조·절차·항목 자체가 PEP에 없으면 완전성 위반입니다.
- 구조·항목은 있으나 내용이 모호하면 검증가능성 위반입니다.
- 둘 다 해당하도록 설계한 경우 output.코멘트에서 두 기준을 모두 다룹니다.

4단계. output 작성
- 시스템 프롬프트의 [평가기준 정의], [다기준 평가 규칙], [종합판정 규칙], [근거(코멘트) 작성 규칙], [출력 형식]을 그대로 적용합니다.
- output은 반드시 JSON 객체로 작성합니다.
- output.판정 값은 "충족", "불가", "검토" 중 하나만 사용합니다.
- output.판정 값에는 괄호 설명을 붙이지 않습니다.
- 목표 결함 유형의 세분 명칭(완전불가, 정확불가, 검증불가, 추적불가, 완전검토 등)은 생성 설계용 내부 라벨이며 output.판정에는 노출하지 않습니다.
- 완전불가, 정확불가, 검증불가, 추적불가 → output.판정은 "불가"로 씁니다.
- 완전검토, 정확검토, 검증검토, 추적검토 → output.판정은 "검토"로 씁니다.
- target_criteria가 2개 이상이면 결함 없는 기준도 반드시 "문제없음", "누락 없음", "모순 없음", "판단 대상 아님" 등으로 확인합니다.
- 정확성에서 비교할 기준값 자체가 없는 경우 "판단 대상 아님"으로 구분하고, 억지로 충족/불가로 몰지 않습니다.
- 코멘트는 결함 기준 우선 서술 → 나머지 기준 확인 → 종합판정 근거 순으로 작성합니다.
- 기준이 1개인 경우에는 자연스러운 한 문단 코멘트를 허용합니다.
- 기준이 2개 이상인 경우에는 기준별 번호 형식을 권장합니다.
- 코멘트에는 "종합적으로 ○○ 기준에서 결함이 확인되므로 최종판정 규칙에 따라 불가로 판정함" 또는 "종합적으로 결함이 확인되지 않아 충족으로 판정함"과 같은 종합 문장을 포함합니다.

5단계. 자체 검증
- output.판정 값이 "충족", "불가", "검토" 중 하나인지 확인합니다.
- output.판정 값에 괄호 설명이 붙어 있지 않은지 확인합니다.
- output.판정 값에 "완전불가", "정확불가", "검증불가", "추적불가", "완전검토", "정확검토", "검증검토", "추적검토" 같은 세분 라벨이 노출되지 않았는지 확인합니다.
- output.코멘트에 등장하는 모든 인용 수치·문구가 rfp_excerpt 또는 pep_excerpt 원문에 실제로 존재하는지 확인합니다.
- description의 설명 문구, rfp_mapping_note, standard_structure를 실제 원문 근거처럼 인용하지 않았는지 확인합니다.
- input.description이 문자열이 아니라 객체 구조인지 확인합니다.
- input.description이 target_description과 동일한 객체 구조로 출력되었는지 확인합니다.
- input.description.id가 평가 대상 PEP 코드인지 확인합니다.
- input.description.rfp_mapping이 대조 대상 RFP 코드인지 확인합니다.
- target_criteria가 target_description.quality 안에 모두 포함되어 있는지 확인합니다.
- target_description에 없는 평가기준, RFP 코드, PEP 코드를 임의로 추가하지 않았는지 확인합니다.
- 평가기준명이 "완전성", "정확성", "검증가능성", "추적성" 중 하나로만 작성되었는지 확인합니다.
- "검증"이라고 쓴 항목이 남아 있으면 "검증가능성"으로 수정합니다.
- "완전"이라고 쓴 항목이 남아 있으면 "완전성"으로 수정합니다.
- "정확"이라고 쓴 항목이 남아 있으면 "정확성"으로 수정합니다.
- 완전성/검증가능성이 동시에 요청된 경우 구조 부재와 표현 모호를 혼동하지 않았는지 확인합니다.
- 추적성 예시라면 Task, 일정, 산출물 중 어느 연결이 끊겼는지 코멘트에 명확히 드러나는지 확인합니다.
- 목표 결함 유형과 실제 설계한 결함이 논리적으로 일치하는지 확인합니다.
- 불일치가 있으면 해당 예시는 폐기하고 다시 생성합니다.

[출력 스키마]
반드시 아래 형태의 JSON 객체 하나만 반환한다. 다른 텍스트·설명·마크다운을 절대 포함하지 않는다.
{
  "items": [
    {
      "id": "PEP-SEED-XX",
      "input": {
        "description": {
          "id": ["PEP-XX"],
          "section_title": "...",
          "description": "...",
          "standard_structure": ["..."],
          "quality": ["완전성"],
          "rfp_mapping": ["RFP-XX"],
          "rfp_mapping_note": "..."
        },
        "rfp_excerpt": "...",
        "pep_excerpt": "..."
      },
      "output": {
        "판정": "충족",
        "코멘트": "..."
      }
    }
  ]
}

[생성 개수]
- 1회 호출당 N개를 생성합니다. 기본값은 5개입니다.
- target_description, target_criteria, target_defect_type이 서로 겹치지 않도록 다양화합니다.
- 기존 분포표를 참고하되, 매트릭스 검증은 프롬프트가 아니라 코드의 PEP_DESCRIPTION_MAP과 target_description으로 처리합니다.
- 충족·불가·검토 라벨 균형을 맞춥니다.
- 추적성 검토 유형 중 "일정/산출물 섹션 자체가 입력에 없는" 케이스도 우선 생성 대상에 포함합니다."""

In [20]:
def build_user_message(desc_code, target_criteria, target_defect_type, existing_ids, n=BATCH_N):
    payload = {
        "target_description": PEP_DESCRIPTION_MAP[desc_code],
        "target_criteria": target_criteria,
        "target_defect_type": target_defect_type,
        "existing_ids": sorted(existing_ids),
        "n": n,
    }
    return json.dumps(payload, ensure_ascii=False)


def call_generator(desc_code, target_criteria, target_defect_type, existing_ids, n=BATCH_N, retries=3):
    user_msg = build_user_message(desc_code, target_criteria, target_defect_type, existing_ids, n)
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=0.7,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
            )
            raw = resp.choices[0].message.content
            parsed = json.loads(raw)
            items = parsed.get("items", [])
            if isinstance(items, list) and items:
                return items
        except Exception as e:
            print(f"  [warn] 호출 실패(attempt {attempt+1}): {e}")
            time.sleep(2)
    return []

In [21]:
VALID_VERDICTS = {"충족", "불가", "검토"}
BANNED_SUBLABELS = ["완전불가", "정확불가", "검증불가", "추적불가", "완전검토", "정확검토", "검증검토", "추적검토"]
VALID_CRITERIA_NAMES = {"완전성", "정확성", "검증가능성", "추적성"}

def validate_item(item, desc_code, target_criteria, existing_ids):
    desc = PEP_DESCRIPTION_MAP[desc_code]

    if not isinstance(item, dict) or not {"id", "input", "output"}.issubset(item.keys()):
        return False, "최상위 필드 누락"
    if item['id'] in existing_ids:
        return False, "id 중복"

    inp = item.get('input', {})
    if not isinstance(inp, dict) or not {"description", "rfp_excerpt", "pep_excerpt"}.issubset(inp.keys()):
        return False, "input 필드 누락"

    d = inp['description']
    if not isinstance(d, dict):
        return False, "input.description이 객체가 아님"
    if d.get('id') != desc['id']:
        return False, "input.description.id가 target_description과 불일치"
    if set(d.get('rfp_mapping', [])) - set(desc.get('rfp_mapping', [])):
        return False, "target_description.rfp_mapping에 없는 RFP 코드 사용"
    if not set(target_criteria).issubset(set(desc.get('quality', []))):
        return False, "target_criteria가 quality에 포함되지 않음"

    out = item.get('output', {})
    if not isinstance(out, dict) or not {"판정", "코멘트"}.issubset(out.keys()):
        return False, "output 필드 누락"

    verdict = out['판정']
    if verdict not in VALID_VERDICTS:
        return False, f"판정 값 이상함: {verdict}"
    if "(" in verdict or ")" in verdict:
        return False, "판정 값에 괄호 포함"

    for term in BANNED_SUBLABELS:
        if term == verdict:
            return False, f"판정 필드에 세분 라벨 노출: {verdict}"

    return True, "ok"


def numeric_grounding_check(item):
    """코멘트에 등장하는 숫자/버전 패턴이 rfp_excerpt+pep_excerpt 원문에 있는지 러프하게 확인 (경고용, 자동폐기 아님)"""
    inp = item['input']
    source_text = inp.get('rfp_excerpt', '') + inp.get('pep_excerpt', '')
    comment = item['output'].get('코멘트', '')
    tokens = re.findall(r'[A-Za-z]+[\s\-]?\d+(?:\.\d+)?%?|\d+(?:\.\d+)?%|\d+(?:일|시간|명|개|분|년|건)', comment)
    missing = [t for t in tokens if t not in source_text]
    return missing

In [22]:
augmented = []
call_count = 0
combo_idx = 0

def current_total():
    return len(seed_data) + len(augmented)

while current_total() < TARGET_TOTAL and call_count < MAX_CALLS:
    combo = combos[combo_idx % len(combos)]
    combo_idx += 1
    desc_code = combo['desc_code']
    target_criteria = combo['target_criteria']
    target_defect_type = combo['target_defect_type']

    remaining = TARGET_TOTAL - current_total()
    n = min(BATCH_N, remaining)
    print(f"[{call_count+1}] {desc_code} / {target_criteria} / {target_defect_type} / 요청 {n}개 (누적 {current_total()}/{TARGET_TOTAL})")

    items = call_generator(desc_code, target_criteria, target_defect_type, existing_ids, n=n)
    call_count += 1

    accepted = 0
    for item in items:
        ok, reason = validate_item(item, desc_code, target_criteria, existing_ids)
        if not ok:
            print(f"    폐기: {item.get('id','?')} - {reason}")
            continue
        missing = numeric_grounding_check(item)
        if missing:
            print(f"    [주의] {item.get('id','?')} - 코멘트 인용 수치가 원문에서 안 보임(수동확인 필요): {missing}")
        augmented.append(item)
        existing_ids.add(item['id'])
        accepted += 1
    print(f"    -> {accepted}/{len(items)}개 채택")

    with open('augmented_progress.json', 'w', encoding='utf-8') as f:
        json.dump(augmented, f, ensure_ascii=False, indent=2)

    time.sleep(1)

print(f"\n최종 생성: {len(augmented)}개 (전체 {current_total()}/{TARGET_TOTAL})")

[1] PEP-12 / ['검증가능성', '정확성', '완전성'] / 완전불가 / 요청 5개 (누적 30/150)
    -> 5/5개 채택
[2] PEP-05-02 / ['완전성'] / 충족 / 요청 5개 (누적 35/150)
    -> 5/5개 채택
[3] PEP-13 / ['검증가능성', '완전성'] / 검증검토 / 요청 5개 (누적 40/150)
    -> 5/5개 채택
[4] PEP-10 / ['완전성'] / 충족 / 요청 5개 (누적 45/150)
    -> 5/5개 채택
[5] PEP-04-01 / ['완전성'] / 완전불가 / 요청 5개 (누적 50/150)
    -> 5/5개 채택
[6] PEP-10 / ['완전성'] / 완전불가 / 요청 5개 (누적 55/150)
    -> 5/5개 채택
[7] PEP-05-01 / ['완전성'] / 완전불가 / 요청 5개 (누적 60/150)
    -> 5/5개 채택
[8] PEP-05-02 / ['완전성'] / 완전검토 / 요청 5개 (누적 65/150)
    -> 5/5개 채택
[9] PEP-12 / ['검증가능성', '정확성', '완전성'] / 검증불가 / 요청 5개 (누적 70/150)
    -> 5/5개 채택
[10] PEP-13 / ['검증가능성', '완전성'] / 완전검토 / 요청 5개 (누적 75/150)
    -> 5/5개 채택
[11] PEP-05-01 / ['완전성'] / 충족 / 요청 5개 (누적 80/150)
    -> 5/5개 채택
[12] PEP-14 / ['검증가능성', '정확성', '완전성'] / 완전불가 / 요청 5개 (누적 85/150)
    -> 5/5개 채택
[13] PEP-15 / ['완전성', '정확성'] / 정확검토 / 요청 5개 (누적 90/150)
    -> 5/5개 채택
[14] PEP-12 / ['검증가능성', '정확성', '완전성'] / 정확불가 / 요청 5개 (누적 95/150)
    -> 5/5개 채택
[15] PEP-11-04 

In [23]:
final_dataset = seed_data + augmented
with open('PEP_AUGMENTED_150.json', 'w', encoding='utf-8') as f:
    json.dump(final_dataset, f, ensure_ascii=False, indent=2)

print(f"최종 데이터셋 크기: {len(final_dataset)}개")
from collections import Counter
print("판정 분포:", Counter(d['output']['판정'] for d in final_dataset))
print("description.id 조합 분포:")
for k, v in Counter(tuple(d['input']['description']['id']) for d in final_dataset).items():
    print(' ', k, v)

최종 데이터셋 크기: 150개
판정 분포: Counter({'불가': 62, '검토': 45, '충족': 43})
description.id 조합 분포:
  ('PEP-03-01', 'PEP-03-02') 2
  ('PEP-04-01', 'PEP-04-02', 'PEP-04-03') 4
  ('PEP-05-01', 'PEP-05-02') 2
  ('PEP-06',) 2
  ('PEP-07', 'PEP-06') 4
  ('PEP-10',) 12
  ('PEP-11-01', 'PEP-11-02', 'PEP-11-03', 'PEP-11-04') 3
  ('PEP-12',) 28
  ('PEP-13',) 12
  ('PEP-14',) 14
  ('PEP-15',) 12
  ('PEP-05-02',) 10
  ('PEP-04-01',) 5
  ('PEP-05-01',) 10
  ('PEP-11-04',) 5
  ('PEP-11-03',) 5
  ('PEP-11-01',) 5
  ('PEP-11-02',) 5
  ('PEP-03-01',) 5
  ('PEP-16',) 5
